## Shared Setup

The code uses a dependency-light stack: `pandas`, `numpy`, and the standard library. 


In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd


DATA_CANDIDATES = [
    Path('/Users/heshuhua/Desktop/Antartica/data.xlsx'),
    Path('Location 2'),
    Path('Location 3')
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data.xlsx.")
print(DATA_PATH)


def ols(y, X):
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)
    if X.size == 0:
        X = np.empty((len(y), 0))
    elif X.ndim == 1:
        X = X.reshape(-1, 1)
    design = np.column_stack([np.ones(len(y)), X])
    beta = np.linalg.pinv(design.T @ design) @ design.T @ y
    fitted = design @ beta
    resid = y - fitted
    n, p = design.shape
    sse = float(resid @ resid)
    tss = float(((y - y.mean()) @ (y - y.mean())))
    r2 = 1 - sse / tss
    sigma2 = sse / (n - p)
    cov = sigma2 * np.linalg.pinv(design.T @ design)
    se = np.sqrt(np.diag(cov))
    return {"beta": beta, "fitted": fitted, "resid": resid, "se": se, "t": beta/se,
            "r2": r2, "adj_r2": 1 - (1-r2)*(n-1)/(n-p), "rmse": math.sqrt(sse/n),
            "aic": n*math.log(sse/n)+2*p, "bic": n*math.log(sse/n)+p*math.log(n)}


def forward_select(y, X_df, criterion="aic"):
    selected, remaining = [], list(X_df.columns)
    best = ols(y, np.empty((len(y), 0)))[criterion]
    while remaining:
        score, col = min((ols(y, X_df[selected+[c]].to_numpy())[criterion], c) for c in remaining)
        if score >= best:
            break
        selected.append(col)
        remaining.remove(col)
        best = score
    return selected


def expanding_validation(y, X, min_train=60):
    pred, actual = [], []
    for i in range(min_train, len(y)):
        beta = ols(y[:i], X[:i])["beta"]
        pred.append(np.r_[1, X[i]] @ beta)
        actual.append(y[i])
    pred, actual = np.asarray(pred), np.asarray(actual)
    return np.sqrt(np.mean((actual-pred)**2)), 1 - np.sum((actual-pred)**2)/np.sum((actual-actual.mean())**2)


def performance_stats(r, periods=12):
    r = np.asarray(r, dtype=float)
    wealth = np.cumprod(1 + r)
    dd = wealth / np.maximum.accumulate(wealth) - 1
    return {"monthly_mean": r.mean(), "annual_return": np.prod(1+r)**(periods/len(r))-1,
            "annual_volatility": r.std(ddof=1)*math.sqrt(periods),
            "sharpe": r.mean()/r.std(ddof=1)*math.sqrt(periods),
            "max_drawdown": dd.min(), "var_5pct": np.quantile(r, 0.05)}


/Users/heshuhua/Desktop/Antartica/data.xlsx


## Question 2

I clean impossible monthly return observations, select a parsimonious factor model, compare the fund to a factor-mimicking portfolio, and test whether rolling betas look stable.


In [4]:
returns = pd.read_excel(DATA_PATH, sheet_name="returns data")
target = "Hedge Fund"
factor_cols = [c for c in returns.columns if c not in ["perf_date", target]]

bad = (returns[factor_cols].abs() > 1).any(axis=1) | (returns[target].abs() > 1)
clean = returns.loc[~bad].reset_index(drop=True)

y = clean[target].to_numpy(float)
X_all = clean[factor_cols].loc[:, clean[factor_cols].std() > 1e-12]
selected = forward_select(y, X_all, "aic")
X = clean[selected].to_numpy(float)
model = ols(y, X)
oos_rmse, oos_r2 = expanding_validation(y, X)

coef_table = pd.DataFrame({
    "term": ["alpha"] + selected,
    "coef": model["beta"],
    "std_error": model["se"],
    "t_stat": model["t"],
    "normal_approx_p": [math.erfc(abs(t)/math.sqrt(2)) for t in model["t"]],
})

print(f"Rows kept: {len(clean)} of {len(returns)}")
print("Dropped dates:", list(returns.loc[bad, "perf_date"].dt.date))
print("Selected factors:", selected)
print(f"R^2={model['r2']:.3f}, adjusted R^2={model['adj_r2']:.3f}, RMSE={model['rmse']:.4f}")
print(f"Expanding-window RMSE={oos_rmse:.4f}, OOS R^2={oos_r2:.3f}\n")
print(coef_table.to_string(index=False))

factor_portfolio = X @ model["beta"][1:]
comparison = pd.DataFrame([performance_stats(y), performance_stats(factor_portfolio)], index=["fund", "factor_portfolio"])
print("\n", comparison.to_string(float_format=lambda x: f"{x:,.4f}"))

rolling = []
for end in range(36, len(clean)+1):
    rolling.append(ols(y[end-36:end], X[end-36:end])["beta"])
rolling = np.asarray(rolling)
stationarity = pd.DataFrame({
    "term": ["alpha"] + selected,
    "mean": rolling.mean(axis=0),
    "std": rolling.std(axis=0, ddof=1),
    "min": rolling.min(axis=0),
    "max": rolling.max(axis=0),
    "first_rolling": rolling[0],
    "last_rolling": rolling[-1],
})
print("\n", stationarity.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))


Rows kept: 189 of 195
Dropped dates: [datetime.date(2008, 1, 31), datetime.date(2008, 2, 29), datetime.date(2008, 3, 31), datetime.date(2020, 12, 31), datetime.date(2021, 1, 31), datetime.date(2021, 2, 28)]
Selected factors: ['Factor - Value vs Growth', 'Factor - Credit', 'Factor - Emerging Markets']
R^2=0.287, adjusted R^2=0.276, RMSE=0.0242
Expanding-window RMSE=0.0245, OOS R^2=0.330

                     term      coef  std_error    t_stat  normal_approx_p
                    alpha  0.008914   0.001795  4.966342     6.822755e-07
 Factor - Value vs Growth -0.562833   0.068119 -8.262527     1.426210e-16
          Factor - Credit  0.154882   0.065546  2.362940     1.813059e-02
Factor - Emerging Markets -0.156015   0.085734 -1.819749     6.879728e-02

                   monthly_mean  annual_return  annual_volatility  sharpe  max_drawdown  var_5pct
fund                    0.0096         0.1164             0.0997  1.1589       -0.1401   -0.0357
factor_portfolio        0.0007         0.007

### Q2 Answer

The preferred cleaned model is:

\[
\hat r_t =
0.008914
-0.562833F_{\text{Value vs Growth},t}
+0.154882F_{\text{Credit},t}
-0.156015F_{\text{Emerging Markets},t}.
\]

The fund has positive alpha, stronger profitability than the factor portfolio, and higher volatility. Rolling betas are not stable enough to treat as stationary.


## Question 3

I use parametric distributions only: lognormal for wait time, Poisson for visitor count, a conditional lognormal model for wait time given visitors, and stylized return diagnostics for tick-data authenticity.


In [5]:
def normal_cdf(z):
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

def poisson_cdf(k, lam):
    p = math.exp(-lam)
    total = p
    for i in range(1, k+1):
        p *= lam / i
        total += p
    return total

def poisson_pmf(max_k, lam):
    p = [math.exp(-lam)]
    for x in range(max_k):
        p.append(p[-1] * lam / (x + 1))
    return np.asarray(p)

def aic(ll, k):
    return 2*k - 2*ll

website = pd.read_excel(DATA_PATH, sheet_name="website data")
wait = website["estimated wait time (mins)"].to_numpy(float)
visitors = website["number of visitors"].to_numpy(int)

log_wait = np.log(wait)
w_mu, w_var = wait.mean(), wait.var(ddof=0)
log_mu, log_sigma = log_wait.mean(), log_wait.std(ddof=0)
gamma_shape, gamma_scale = w_mu**2 / w_var, w_var / w_mu

wait_fits = pd.DataFrame([
    ("lognormal", aic(np.sum(-np.log(wait*log_sigma*math.sqrt(2*math.pi)) - (log_wait-log_mu)**2/(2*log_sigma**2)), 2)),
    ("gamma_mom", aic(np.sum((gamma_shape-1)*np.log(wait) - wait/gamma_scale - gamma_shape*np.log(gamma_scale) - math.lgamma(gamma_shape)), 2)),
    ("normal", aic(np.sum(-0.5*np.log(2*math.pi*w_var) - (wait-w_mu)**2/(2*w_var)), 2)),
    ("exponential", aic(np.sum(np.log(1/w_mu) - wait/w_mu), 1)),
], columns=["distribution", "aic"]).sort_values("aic")

lam = visitors.mean()
v_var = visitors.var(ddof=0)
pois_ll = sum(x*math.log(lam) - lam - math.lgamma(x+1) for x in visitors)
norm_ll = np.sum(-0.5*np.log(2*math.pi*v_var) - (visitors-lam)**2/(2*v_var))
visitor_fits = pd.DataFrame([("poisson", aic(pois_ll, 1)), ("normal_approx", aic(norm_ll, 2))], columns=["distribution", "aic"]).sort_values("aic")

p_wait_gt_10 = 1 - normal_cdf((math.log(10) - log_mu) / log_sigma)
p_visitors_lt_46 = poisson_cdf(45, lam)

log_wait_model = ols(np.log(wait), visitors)
intercept, slope = log_wait_model["beta"]
sigma = log_wait_model["resid"].std(ddof=2)
pmf = poisson_pmf(250, lam)
p_cond = sum((1-normal_cdf((math.log(7.5)-(intercept+slope*n))/sigma))*p for n,p in enumerate(pmf) if n >= 50) / pmf[50:].sum()

print(wait_fits.to_string(index=False))
print()
print(visitor_fits.to_string(index=False))
print(f"\nP(wait > 10) = {p_wait_gt_10:.4f}")
print(f"P(visitors < 46) = {p_visitors_lt_46:.4f}")
print(f"corr(wait, visitors) = {np.corrcoef(wait, visitors)[0,1]:.3f}")
print(f"log(wait) = {intercept:.4f} + {slope:.4f} * visitors + error")
print(f"P(wait > 7.5 | visitors >= 50) = {p_cond:.4f}")


distribution          aic
   lognormal 44616.075571
   gamma_mom 45189.257135
      normal 48639.456039
 exponential 52025.076567

 distribution          aic
      poisson 68913.677131
normal_approx 68943.151687

P(wait > 10) = 0.0538
P(visitors < 46) = 0.0325
corr(wait, visitors) = 0.692
log(wait) = -1.5101 + 0.0502 * visitors + error
P(wait > 7.5 | visitors >= 50) = 0.1636


In [6]:
def autocorr(x, lag=1):
    x = np.asarray(x, dtype=float)
    a = x[:-lag] - x[:-lag].mean()
    b = x[lag:] - x[lag:].mean()
    return float((a @ b) / np.sqrt((a @ a) * (b @ b)))

ticks = pd.read_excel(DATA_PATH, sheet_name="tick data")
rows = []
for asset in ticks.columns:
    x = ticks[asset].to_numpy(float)
    z = (x - x.mean()) / x.std(ddof=0)
    rows.append({
        "asset": asset,
        "skew": np.mean(z**3),
        "excess_kurtosis": np.mean(z**4) - 3,
        "tail_abs_z_gt_3": np.mean(np.abs(z) > 3),
        "abs_return_acf_1": autocorr(np.abs(x), 1),
        "abs_return_acf_5": autocorr(np.abs(x), 5),
        "zero_return_share": np.mean(x == 0),
    })
features = pd.DataFrame(rows)
features["real_score"] = (
    features["excess_kurtosis"].clip(lower=0)
    + 20 * features[["abs_return_acf_1", "abs_return_acf_5"]].mean(axis=1).clip(lower=0)
    + 50 * (features["tail_abs_z_gt_3"] - 0.0027).clip(lower=0)
)
features["classification"] = np.where(features["real_score"] > 1, "real", "simulated")
print(features.sort_values("real_score", ascending=False).to_string(index=False, float_format=lambda x: f"{x:,.4f}"))


  asset    skew  excess_kurtosis  tail_abs_z_gt_3  abs_return_acf_1  abs_return_acf_5  zero_return_share  real_score classification
Asset F -1.3988          48.2160           0.0150            0.2441            0.2270             0.0237     53.5399           real
Asset B  0.1716           5.8511           0.0146            0.2286            0.1861             0.0221     10.5917           real
Asset C  0.0281           4.9095           0.0155            0.2448            0.2201             0.0462     10.1991           real
Asset D  0.0301           4.9348           0.0160            0.2192            0.1877             0.0065      9.6670           real
Asset G  0.0832           3.9969           0.0141            0.2501            0.2364             0.0898      9.4310           real
Asset A  0.0184          -0.0314           0.0026            0.0057           -0.0012             0.0000      0.0448      simulated
Asset E  0.0043           0.0363           0.0028            0.0027         

### Q3 Answer

\[
W\sim\operatorname{LogNormal}(1.4666,0.5195),\qquad
N\sim\operatorname{Poisson}(59.2909).
\]

\[
P(W>10)=0.0538,\qquad P(N<46)=0.0325,\qquad
P(W>7.5\mid N\ge50)=0.1636.
\]

Likely real assets:

\[
\boxed{B,C,D,F,G}
\]

Likely simulated assets:

\[
\boxed{A,E,H,I,J}
\]
